# Shock Chemistry with UCLCHEM

In this tutorial, we explore how shocks affect the physical and chemical evolution of interstellar gas using UCLCHEM.

We will:
- build intuition for C-type and J-type shocks
- examine how physical conditions evolve
- explore how key parameters influence the chemistry

The goal is to become comfortable running and interpreting shock models.

## 0. Setup

In this section, we prepare the environment and make sure everything runs correctly.

👉 There is no need to modify anything here.

In [ ]:
import uclchem
import matplotlib.pyplot as plt

## Step 1: Pre-shock evolution (Stage 1)

### Goal
Understand the physical and chemical state of the gas before the shock occurs.

### What happens in this phase?
- The gas is cold and dense
- Chemistry evolves over time
- Molecules form in the gas and freeze onto dust grains
- Ice mantles build up

👉 This phase defines the **initial conditions** for the shock.

In [ ]:
# For our convencience we will define the "global" initial temperature. 
# It will be useful later when we will play around with the post-shock temperature.
global_initial_temp  = 10.0 
# To make our modeling easier, we will also define here the density of the medium in which the shock will propagate.
# In the stage 1 modeling we start from "diffuse" cloud and collapse it to a chosen density. This chosen (final) density
# is the density of our pre-shock medium. 
pre_shock_density    = 1e3

param_dict = {
                "endAtFinalDensity":    False,                        # stop once the density is reached or at finalTime
                "freefall":             True,                         # turn on a collapse in free-fall
                "initialDens":          1e2,                          # starting density of this stage
                "finalDens":            pre_shock_density,            # final density of this stage
                "initialTemp":          global_initial_temp,          # temperature of gas and dust
                "finalTime":            6e+06,                        # final time of the stage
                "rout":                 0.5,                          # radius of cloud in pc
                "baseAv":               2.0,                          # visual extinction at cloud edge
                "zeta":                 1e0,                          # Cosmic ray ionisation rate factor
                "radfield":             1e1,                          # radiation field factor
             }

preshock    = uclchem.model.Cloud(param_dict = param_dict)
df_preshock = preshock.get_dataframes()
preshock.check_conservation()

Before inspecting the results, we define helper plotting functions that will be used throughout the tutorial.

👉 You do not need to modify these functions.

In [ ]:
def plot_physics(
    y1,
    y1_label,
    y2,
    y2_label,
    data,
    xscale="linear",
    y1scale="linear",
    y2scale="linear",
    title=None,
):
    fig, ax1 = plt.subplots(figsize=(7, 4.5))

    ax1.plot(data["Time"], data[y1], lw=2, label=y1_label)
    ax1.set_xlabel("Time [yr]")
    ax1.set_ylabel(y1_label)
    ax1.set_xscale(xscale)
    ax1.set_yscale(y1scale)
    ax1.grid(alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(data["Time"], data[y2], lw=2, linestyle="--", label=y2_label)
    ax2.set_ylabel(y2_label)
    ax2.set_yscale(y2scale)
    
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper left")

    if title is not None:
        ax1.set_title(title)
        
    xmax = data["Time"].max()
    tmin = data["Time"][data["Time"] > 0].min()
    ax1.set_xlim(tmin, xmax)

    fig.tight_layout()
    plt.show()
    
def plot_abundances(
    species_list,
    data,
    title=None,
    ymin=1e-14,
    ymax=1e-3,
    xmax=None,
):
    fig, ax = plt.subplots(figsize=(7, 4.5))

    for species in species_list:
        ax.plot(data["Time"], data[species], lw=2, label=species)

    ax.set_xlabel("Time [yr]")
    ax.set_ylabel("Abundance")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(ymin, ymax)

    if xmax is None:
        xmax = data["Time"].max()
    tmin = data["Time"][data["Time"] > 0].min()
    ax.set_xlim(tmin, xmax)

    ax.grid(alpha=0.3, which="both")
    ax.legend(frameon=False)

    if title is not None:
        ax.set_title(title)

    fig.tight_layout()
    plt.show()
    
def compare_runs(
    data_list,
    labels,
    quantity,
    ylabel,
    title=None,
    xscale="log",
    yscale="linear",
):
    fig, ax = plt.subplots(figsize=(7, 4.5))

    for df, label in zip(data_list, labels):
        ax.plot(df["Time"], df[quantity], lw=2, label=label)

    ax.set_xlabel("Time [yr]")
    ax.set_ylabel(ylabel)
    ax.set_xscale(xscale)
    ax.set_yscale(yscale)
    ax.grid(alpha=0.3, which="both")
    ax.legend(frameon=False)

    if title is not None:
        ax.set_title(title)

    fig.tight_layout()
    plt.show()
    
def compare_species_across_runs(
    species_list,
    data_list,
    labels,
    ymin=1e-14,
    ymax=1e-3,
    title_prefix="",
):
    for sp in species_list:
        fig, ax = plt.subplots(figsize=(7, 4.5))

        for df, label in zip(data_list, labels):
            ax.plot(df["Time"], df[sp], lw=2, label=label)

        ax.set_xlabel("Time [yr]")
        ax.set_ylabel(f"{sp} abundance")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_ylim(ymin, ymax)
        ax.grid(alpha=0.3, which="both")
        ax.legend(frameon=False)
        ax.set_title(f"{title_prefix}{sp}")

        fig.tight_layout()
        plt.show()

### Questions

- Which species is most abundant?
- Which species shows the most variation over time?
- Why is this phase important before introducing a shock?

👉 Take a moment to look at the trends before moving on.

### Physical evolution

Let’s first look at how the physical conditions evolve.

👉 Pay attention to:
- temperature
- density

In [ ]:
plot_physics(
             y1       = "gasTemp",
             y1_label = "Temperature [K]",
             y2       = "Density",
             y2_label = "Density [cm$^{-3}$]",
             data     = df_preshock,
             xscale   = "log",
             y1scale  = "log",
             y2scale  = "log",
             title    = "Pre-shock: temperature and density evolution",
)

### Chemical evolution

Now we examine how molecular abundances evolve during the pre-shock phase.

We select a few species of interest and track their abundances over time.

In [ ]:
plot_abundances(
    species_list = ["SIO", "CH3OH", "CO"],
    data         = df_preshock,
    title        = "Pre-shock: gas-phase molecular abundances",
)

## Step 2: Shock propagation (Stage 2)

In this stage, a shock propagates through the medium.

### What changes compared to the pre-shock phase?
- Temperature increases
- Density increases
- Grain processing releases material into the gas phase

👉 We will explore two types of shocks:
- C-type (continuous)
- J-type (jump)

### ⚠️ Important: choosing physically consistent parameters

UCLCHEM uses a parametrized description of shocks rather than solving the full MHD equations.

This means the shock type is not determined self-consistently by the code (yet).

Instead, it is set by the input parameters.

In particular, the parameter `bm0` controls the magnetic field strength and must be chosen so that the model is consistent with a C-type shock.

👉 In this tutorial, we keep `bm0` fixed and focus on the effects of density and velocity.

In practice, `bm0` is often chosen based on values used in previous shock models or calibrated MHD simulations, rather than being freely varied.

## Step 2a: Modeling a C-type shock

### Goal
Understand how a continuous shock affects the physical and chemical evolution.

In a **C-type shock**:
- physical quantities change smoothly
- the gas is heated and compressed
- grain processing (sputtering) injects new species into the gas

👉 Run the model below and inspect the results.

In [ ]:
param_dict["initialDens"] = pre_shock_density # now, our initialDensity is no longer 1e2, but the density to which we colllapsed the cloud, i.e., final density
param_dict["bm0"]         = 1                 # magnetic parameter
param_dict["finalTime"]   = 1e6               # the length of this shock stage [yr]
param_dict["freefall"]    = False             # the cloud no longer collapses, so we have to turn this off (it is also done by default, but we can be precise here)
shock_vel                 = 10                # shock velocity; for now we will use the same for both shock types


cshock = (
    uclchem.model.CShock(
                         shock_vel        = shock_vel, 
                         param_dict       = param_dict, 
                         previous_model   = preshock
                        )
    )

df_cshock = cshock.get_dataframes()
cshock.check_conservation()

### Inspecting the results

Let’s examine how the physical and chemical properties evolve during the C-shock.

👉 Focus on:
- how temperature changes
- how density evolves
- how abundances respond

In [ ]:
plot_physics(
             y1       = "gasTemp",
             y1_label = "Temperature [K]",
             y2       = "Density",
             y2_label = "Density [cm$^{-3}$]",
             data     = df_cshock,
             xscale   = "log",
             y1scale  = "log",
             y2scale  = "log",
             title    = "C-shock: temperature and density evolution",
            )

In [ ]:
plot_abundances(
    species_list = ["SIO", "CH3OH", "CO"],
    data         = df_cshock,
    title        = "C-shock: gas-phase molecular abundances",
)

### Questions

- Which species shows the strongest response to the shock?
- At what stage do abundance peaks occur?
- How does this compare to the pre-shock phase?

In [ ]:
# Here you can extract all the additional information if you wish, using operations on dataframes, e.g., df_cshock['gasTemp].max()

## Step 2b: Modeling a J-type shock

### Goal
Compare a J-type shock with the C-type shock.

In a **J-type shock**:
- physical conditions change abruptly
- temperatures are typically higher
- the environment is more destructive

👉 Run the model and compare it with the C-shock.

In [ ]:
jshock = (
    uclchem.model.JShock(
                         shock_vel      = shock_vel, 
                         param_dict     = param_dict, 
                         previous_model = preshock,
                         timepoints     = 1500
                        )
    )

df_jshock = jshock.get_dataframes()
jshock.check_conservation()

### Inspecting the J-shock

Compare the results with the C-type shock.

In [ ]:
plot_physics(
             y1         = "gasTemp",
             y1_label   = "Temperature [K]",
             y2         = "Density",
             y2_label   = "Density [cm$^{-3}$]",
             data       = df_jshock,
             xscale     = "log",
             y1scale    = "log",
             y2scale    = "log",
             title      = "J-shock: temperature and density evolution",
            )

In [ ]:
plot_abundances(
    species_list = ["SIO", "CH3OH", "CO"],
    data         = df_jshock,
    title        = "J-shock: gas-phase molecular abundances",
)

### Questions

- Which shock reaches higher temperatures?
- Which shows more abrupt changes?
- How do the abundances differ between the two models?

👉 Identify one key difference between C-type and J-type shocks.

In [ ]:
# Here you can extract all the additional information if you wish, using operations on dataframes, e.g., df_jshock['gasTemp].max()

## Step 3: Density and the onset of the C-shock

### Goal
Understand how the pre-shock density affects the shock evolution.

### What changes?
We vary the **initial density** while keeping all other parameters fixed.

👉 Your task:
- Run 3 C-shock models with densities:
  - 1e3
  - 1e4
  - 1e5

However, feel free to choose other densities as well to get a better feeling of how properties of the medium change. 

In [ ]:
# run models for different densities

In [ ]:
# save the dataframes, e.g., df_dens_1e3 = dens_1e3.get_dataframes()

In [ ]:
compare_runs(
    data_list = [], # Provide the names to the dataframes
    labels    = [r"$10^3$ cm$^{-3}$", r"$10^4$ cm$^{-3}$", r"$10^5$ cm$^{-3}$"], # Write the labels - each label should describe the dataframes in the exact order you provided them 
    quantity  = "gasTemp",
    ylabel    = "Temperature [K]",
    title     = "Effect of shock velocity on C-shock temperature evolution",
    xscale    = "log",
    yscale    = "log",
)

### Questions

- How does density affect the timing of the shock?
- Do higher densities lead to faster or slower evolution?
- When do abundance peaks occur?

👉 Think about how density influences reaction rates.

## Step 4: Shock velocity and molecular abundances

### Goal
Understand how shock strength affects chemical abundances.

### What changes?
We vary the **shock velocity**, which controls:
- temperature increase
- efficiency of grain processing
- overall shock strength

👉 Keep all other parameters fixed (for the purpose of this test we will fix the density).

### Task

Run the C-shock model for:
- 10 km/s
- 20 km/s
- 30 km/s

Plot the abundances of:
- SiO
- CH3OH
- CO

Again, feel free to adjust the velocities and species. Just constrain yourself to velocities below 45 km/s. 

In [ ]:
# run models for v = 10, 20, 30 km/s
# it's probably best to run these models in different cells or 
# copy-paste the code for different shock velocities, changing only the shock_vel variable
param_dict['initialDens'] = 1e4 # Just an example (we were playing with the densities before, so it is good to choose some value deliverate)

In [ ]:
# save the dataframes, e.g., df_vel_10 = vel_10.get_dataframes()

In [ ]:
# plot selected species (SiO, CH3OH, CO) 
# (feel free to also plot other species, e.g., H2O, OH, H2CO, etc. - these are just examples)
compare_runs(
    data_list = [], # Provide the names to the dataframes
    labels    = [r"$10$ km/s", r"$20$ km/s", r"$30$ km/s"], # Write the labels - each label should describe the dataframes in the exact order you provided them 
    quantity  = "gasTemp",
    ylabel    = "Temperature [K]",
    title     = "Effect of shock velocity on C-shock temperature evolution",
    xscale    = "log",
    yscale    = "log",
)

In [ ]:
compare_species_across_runs(
    species_list   = ["SIO", "CH3OH", "CO"],  # Define the species
    data_list      = [],  # Provide the names to the dataframes
    labels         = [r"$10$ km/s", r"$20$ km/s", r"$30$ km/s"] # Write the labels - each label should describe the dataframes in the exact order you provided them 
)

### Questions

- Which species increases most with velocity?
- Which species remains relatively stable?
- How does velocity affect the timing of abundance peaks?

👉 Which molecule would you choose as a shock tracer?

In [ ]:
# Here you can extract all the additional information if you wish

---
## Summary

In this tutorial, we explored:

- how shocks modify physical conditions (temperature, density)
- the role of grain processing in releasing material into the gas phase
- differences between C-type and J-type shocks
- how key parameters (density, velocity) affect the chemistry

### Key takeaway

Shock chemistry is controlled by a small number of physical parameters, but the chemical response can be complex and species-dependent.

👉 You should now feel comfortable running and interpreting basic shock models in UCLCHEM.